# Phase 2: Exploratory Data Analysis — Combined Scholarship Dataset

**Name:** Julia Lee

**Course:** DSCI 2001-51 Data Science I

**Date:** 11/08/2026

**Dataset:** Combined from seven student-collected workbooks:
- `Kone_Cheick_National_Scholarships.xlsx` (national general scholarships)
- `Kone_Cheick_NH_Scholarships.xlsx` (New Hampshire scholarships)
- `Haji-Said_Yahya_National_Scholarships.xlsx` (national women-focused scholarships)
- `Haji-Said_Yahya_Indiana_Scholarships.xlsx` (Indiana women-focused scholarships)
- `Welzant_Kayla_scholarships_NY_leadership_civic_N-Z.xlsx` (New York leadership/civic scholarships)
- `Lee_Julia_Data.xlsx` (New Mexico STEM scholarships)
- `Jaibrien_Brooks Scholarship Data.xlsx` (California + national teaching scholarships)

---

## Phase 2 Objective

Per the project brief, Phase 2 covers:

- **Clean and standardize** fields such as award amount, grade level, deadline, scholarship type, state, and eligibility
- **Calculate summary statistics and frequency distributions**
- **Identify** missing values, duplicate scholarships, expired listings, and inconsistent categories
- **Perform initial comparisons** across states, grade levels, scholarship categories, award amounts, and deadlines

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Upload All 7 Excel files

files = [
    'Kone_Cheick_National_Scholarships.xlsx',
    'Kone_Cheick_NH_Scholarships.xlsx',
    'Haji-Said_Yahya_National_Scholarships.xlsx',
    'Haji-Said_Yahya_Indiana_Scholarships.xlsx',
    'Welzant_Kayla_scholarships_NY_leadership_civic_N-Z.xlsx',
    'Lee_Julia_Data.xlsx',
    'Jaibrien_Brooks Scholarship Data.xlsx',
]

frames = []
for f in files:
  try:
    sheet = pd.read_excel(f,sheet_name=0)
    sheet['_source_file'] = f
    frames.append(sheet)
    print(f"Loaded: {f:55s} {sheet.shape[0]:>3} rows")
  except Exception as e:
    print(f"Could not be load {f}: {e}")

df = pd.concat(frames, ignore_index=True)
print("\n" + "=" * 70)
print("Combined Dataset Shape:", df.shape)
print("=" * 70)

Loaded: Kone_Cheick_National_Scholarships.xlsx                   25 rows
Loaded: Kone_Cheick_NH_Scholarships.xlsx                         25 rows
Loaded: Haji-Said_Yahya_National_Scholarships.xlsx               18 rows
Loaded: Haji-Said_Yahya_Indiana_Scholarships.xlsx                22 rows
Loaded: Welzant_Kayla_scholarships_NY_leadership_civic_N-Z.xlsx  15 rows
Loaded: Lee_Julia_Data.xlsx                                      23 rows
Loaded: Jaibrien_Brooks Scholarship Data.xlsx                    32 rows

Combined Dataset Shape: (160, 30)


In [2]:
# PHASE 2a: CLEANING & STANDARDIZATION

# 1. Parse award amount into a single numeric max value
def parse_max_value(val):
  if pd.isna(val):
    return np.nan
  s = str(val).replace(',', '').replace('$', '').replace('+', '')
  nums = re.findall(r'\d+', s)
  return float(nums[-1]) if nums else np.nan

df['award_max_num'] = df['Maximum potential value'].apply(parse_max_value)

# 2. Bucket award amounts into standard ranges
def bucket_award(x):
  if pd.isna(x):
    return 'Unknown'
  if x < 1000:
    return 'Under $1,000'
  if x < 5000:
    return '$1,000-$4,999'
  if x < 10000:
      return '$5,000-$9,999'
  return '$10,000 or more'

df['award_bucket'] = df['award_max_num'].apply(bucket_award)

print("- Award Amount Standardization -")
print("Records with parseable value:", df['award_max_num'].notna().sum(), "of", len(df))
print("\nAward distribution:")
print(df['award_bucket'].value_counts())

- Award Amount Standardization -
Records with parseable value: 88 of 160

Award distribution:
award_bucket
Unknown            72
$1,000-$4,999      37
$10,000 or more    26
$5,000-$9,999      17
Under $1,000        8
Name: count, dtype: int64


In [3]:
# Standardize grade level into numeric min/max
def parse_grade_range(val):
  if pd.isna(val):
    return (np.nan, np.nan)
  s = str(val).strip().lower().replace('k', '0')
  nums = re.findall(r'\d+', s)
  if not nums:
    return (np.nan, np.nan)
  if len(nums) == 1:
    return (float(nums[0]), float(nums[0]))
  return (float(nums[0]), float(nums[-1]))

grades = df['Eligible grade levels'].apply(parse_grade_range)
df['grade_min'] = [g[0] for g in grades]
df['grade_max'] = [g[1] for g in grades]

print("- Grade level classification (from template column) -")
print(df['Middle school / high school / both'].value_counts(dropna=False))

print("\n- Grade range examples (min-max) -")
print(df[['Eligible grade levels', 'grade_min', 'grade_max']].dropna().head(10).to_string(index=False))

- Grade level classification (from template column) -
Middle school / high school / both
High school                                     105
NaN                                              22
Both                                              7
High school (seniors) + college                   7
Middle school                                     5
Both (HS seniors) + college                       4
Both (also elementary)                            3
Both (HS seniors) + college/grad                  2
Both (Middle focus)                               1
Unknown                                           1
Middle school (supports through high school)      1
High school + college/graduate                    1
High school + college (level not specified)       1
Name: count, dtype: int64

- Grade range examples (min-max) -
          Eligible grade levels  grade_min  grade_max
         12 / graduating senior       12.0       12.0
         12 / graduating senior       12.0       12.0
         12

In [4]:
# --- Standardize application deadline into a real datetime ---

def parse_deadline(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, (pd.Timestamp, datetime)):
        return pd.Timestamp(val)
    s = str(val).strip()
    if s.lower() in ('unknown', 'varies', '', 'not yet posted', 'not listed'):
        return pd.NaT
    try:
        return pd.to_datetime(s, errors='coerce', fuzzy=True)
    except Exception:
        return pd.NaT

df['deadline_dt'] = df['Application deadline'].apply(parse_deadline)
df['deadline_month'] = df['deadline_dt'].dt.month_name()
df['deadline_season'] = df['deadline_dt'].dt.month.map(
    lambda m: 'Winter (Dec–Feb)' if m in [12, 1, 2] else
              'Spring (Mar–May)' if m in [3, 4, 5] else
              'Summer (Jun–Aug)' if m in [6, 7, 8] else
              'Fall (Sep–Nov)' if m in [9, 10, 11] else 'Unknown'
)

print(" - Deadline standardization - ")
print("Records with parseable deadline:", df['deadline_dt'].notna().sum(), "of", len(df))
print("\nDeadline season distribution:")
print(df['deadline_season'].value_counts())
print("\nDeadline month distribution:")
print(df['deadline_month'].value_counts().head(12))

 - Deadline standardization - 
Records with parseable deadline: 58 of 160

Deadline season distribution:
deadline_season
Unknown             102
Winter (Dec–Feb)     28
Spring (Mar–May)     22
Fall (Sep–Nov)        5
Summer (Jun–Aug)      3
Name: count, dtype: int64

Deadline month distribution:
deadline_month
January     14
March       14
December     7
February     7
April        6
October      5
July         2
May          2
June         1
Name: count, dtype: int64


In [5]:
# Standardize Yes/No/Unknown boolean-like columns

bool_cols = [
    'Financial need required (Yes/No/Unknown)',
    'Essay required',
    'Recommendation required',
    'Transcript/test score required',
    'Community service requirement',
    'Leadership requirement',
    'Renewable (Yes/No/Unknown)',
]

def normalize_yes_no(val):
  if pd.isna(val):
    return 'Unknown'
  s = str(val).strip().lower()
  if re.match(r'^yes\b', s):
    return 'Yes'
  if re.match(r'^no\b', s):
    return 'No'
  return 'Unknown'

for col in bool_cols:
  df[col + '_clean'] = df[col].apply(normalize_yes_no)

# Standardize provider type (fix capitalization)

df['provider_type_clean'] = df['Provider type'].str.strip().str.title()

# Standardize state/region into primary state

def extract_state(val):
  if pd.isna(val):
    return 'Unknown'
  s = str(val).strip()
  if 'National' in s or 'all U.S.' in s or s == 'United States':
    return 'National'
  for state in ['New York', 'New Mexico', 'New Hampshire', 'Indiana',
                'California', 'Texas', 'Tennessee', 'Minnesota',
                'Massachusetts', 'Michigan', 'Iowa', 'Wisconsin',
                'Puerto Rico', 'Canada']:
    if state.lower() in s.lower():
      return state
  return 'Multi-state / Other'

df['state_clean'] = df['State/region'].apply(extract_state)

print(" - Provider type (cleaned) - ")
print(df['provider_type_clean'].value_counts())
print("\n - State (cleaned) - ")
print(df['state_clean'].value_counts())

 - Provider type (cleaned) - 
provider_type_clean
Foundation/Nonprofit                68
Corporation                         17
Private Donor / Online Platform     10
Nonprofit                            8
College/University                   6
Professional Association             5
Government Agency                    3
Community Foundation                 3
Government                           3
Veterans Service Organization        2
Private Donor / Scholarship Fund     2
Corporation / Program                1
Corporation / Foundation             1
Public Benefit Corporation           1
Company                              1
University                           1
Professional Association/Union       1
Corporate/Company Foundation         1
Corporate/Company Program            1
Name: count, dtype: int64

 - State (cleaned) - 
state_clean
National               48
Indiana                23
Unknown                22
New Mexico             20
California             16
New York          

In [6]:
# PHASE 2b: SUMMARY STATISTICS

print("=" * 60)
print("SUMMARY STATISTICS — COMBINED DATASET")
print("=" * 60)

print(f"Total scholarships: {len(df)}")
print(f"Number of source files: {df['_source_file'].nunique()}")
print(f"Number of unique providers: {df['Scholarship provider/organization'].nunique()}")

print("\n - Award amount statistics (parsed) - ")
print(df['award_max_num'].describe().round(2))

print("\n - Award amount by provider type - ")
print(df.groupby('provider_type_clean')['award_max_num'].agg(['count', 'mean', 'median', 'max']).round(0))

print("\n - Award amount by state - ")
print(df.groupby('state_clean')['award_max_num'].agg(['count', 'mean', 'median']).round(0))

print("\n - Award amount by national vs state/local - ")
print(df.groupby('National vs. state/local')['award_max_num'].agg(['count', 'mean', 'median']).round(0))

print("\n - Award amount by source file - ")
print(df.groupby('_source_file')['award_max_num'].agg(['count', 'mean', 'median', 'max']).round(0))

SUMMARY STATISTICS — COMBINED DATASET
Total scholarships: 160
Number of source files: 7
Number of unique providers: 140

 - Award amount statistics (parsed) - 
count        88.00
mean      13321.89
std       22556.32
min         250.00
25%        2000.00
50%        4000.00
75%       13000.00
max      120000.00
Name: award_max_num, dtype: float64

 - Award amount by provider type - 
                                  count      mean    median       max
provider_type_clean                                                  
College/University                    6    1967.0    2400.0    4000.0
Community Foundation                  1    2100.0    2100.0    2100.0
Company                               1     500.0     500.0     500.0
Corporate/Company Foundation          1   50000.0   50000.0   50000.0
Corporate/Company Program             1   55000.0   55000.0   55000.0
Corporation                          17   15765.0    5000.0   80000.0
Corporation / Foundation              1   20000.0   200

In [7]:
# PHASE 2b: FREQUENCY DISTRIBUTIONS

freq_cols = [
    'provider_type_clean',
    'National vs. state/local',
    'Middle school / high school / both',
    'Scholarship category',
    'Renewable (Yes/No/Unknown)_clean',
    'Financial need required (Yes/No/Unknown)_clean',
    'Essay required_clean',
    'Recommendation required_clean',
    'award_bucket',
    'deadline_season',
    'state_clean',
]

for col in freq_cols:
    print(f"\n--- {col} ---")
    counts = df[col].value_counts(dropna=False)
    pcts = (df[col].value_counts(dropna=False, normalize=True) * 100).round(1)
    summary = pd.DataFrame({'count': counts, 'percent': pcts})
    print(summary.head(10))


--- provider_type_clean ---
                                 count  percent
provider_type_clean                            
Foundation/Nonprofit                68     42.5
NaN                                 25     15.6
Corporation                         17     10.6
Private Donor / Online Platform     10      6.2
Nonprofit                            8      5.0
College/University                   6      3.8
Professional Association             5      3.1
Government                           3      1.9
Government Agency                    3      1.9
Community Foundation                 3      1.9

--- National vs. state/local ---
                                  count  percent
National vs. state/local                        
State/local                          83     51.9
National                             64     40.0
Local                                 9      5.6
National (state-level winners)        2      1.2
State                                 1      0.6
National program, 

In [8]:
# PHASE 2c: DATA QUALITY CHECKS

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = df.drop(columns=['_source_file']).isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print("\n - Top 15 most-missing columns - ")
print(missing_summary.head(15))

print("\n - Missingness by source file (percent of cells missing) - ")
def pct_missing(g):
    return round(g.drop(columns='_source_file').isnull().sum().sum() / g.drop(columns='_source_file').size * 100, 1)
missing_by_file = df.groupby('_source_file').apply(pct_missing)
print(missing_by_file.sort_values(ascending=False))

print(f"\nOverall missingness: {df.drop(columns='_source_file').isnull().sum().sum():,} missing cells out of {df.drop(columns='_source_file').size:,} ({df.drop(columns='_source_file').isnull().sum().sum()/df.drop(columns='_source_file').size*100:.1f}%)")
print(f"Columns 100% complete: {(df.drop(columns=['_source_file']).isnull().sum() == 0).sum()} of {len(df.columns) - 1}")

MISSING VALUES

 - Top 15 most-missing columns - 
                                  missing   pct
deadline_dt                           102  63.7
deadline_month                        102  63.7
award_max_num                          72  45.0
grade_min                              58  36.2
grade_max                              58  36.2
Application open date (if known)       45  28.1
Minimum GPA (if any)                   42  26.2
Application deadline                   41  25.6
Number of awards (if known)            33  20.6
Maximum potential value                27  16.9
Field/area of interest                 25  15.6
Transcript/test score required         25  15.6
Provider type                          25  15.6
Renewable (Yes/No/Unknown)             25  15.6
Recommendation required                25  15.6

 - Missingness by source file (percent of cells missing) - 
_source_file
Kone_Cheick_NH_Scholarships.xlsx                           63.2
Kone_Cheick_National_Scholarships.xlsx      

/tmp/ipykernel_25687/3401886255.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_by_file = df.groupby('_source_file').apply(pct_missing)


In [9]:
# PHASE 2c: DUPLICATES & EXPIRED LISTINGS

print("=" * 60)
print("DUPLICATE CHECKS")
print("=" * 60)

print("Exact duplicate rows:", df.duplicated().sum())

df['_name_lower'] = df['Scholarship name'].str.lower().str.strip()
dup_names = df[df['_name_lower'].duplicated(keep=False)].sort_values('_name_lower')
print(f"\nDuplicate scholarship names (case-insensitive): {df['_name_lower'].duplicated().sum()}")
if len(dup_names):
    print(dup_names[['Scholarship name', '_source_file']].to_string(index=False))

name_files = df.groupby('_name_lower')['_source_file'].nunique()
overlaps = name_files[name_files > 1]
print(f"\nScholarships appearing in multiple source files: {len(overlaps)}")
if len(overlaps):
    print(overlaps)

print("\n" + "=" * 60)
print("EXPIRED LISTINGS (deadline in the past, before today)")
print("=" * 60)

today = pd.Timestamp.today().normalize()
df['is_expired'] = df['deadline_dt'].notna() & (df['deadline_dt'] < today)
print(f"Today's date: {today.date()}")
print(f"Records with a deadline: {df['deadline_dt'].notna().sum()}")
print(f"Expired deadlines: {df['is_expired'].sum()}")

if df['is_expired'].sum() > 0:
    expired = df[df['is_expired']][['Scholarship name', 'Application deadline', '_source_file']]
    print("\nExpired records:")
    print(expired.to_string(index=False))

print(f"\nDeadlines still upcoming: {(df['deadline_dt'] >= today).sum()}")

DUPLICATE CHECKS
Exact duplicate rows: 0

Duplicate scholarship names (case-insensitive): 0

Scholarships appearing in multiple source files: 0

EXPIRED LISTINGS (deadline in the past, before today)
Today's date: 2026-11-08
Records with a deadline: 58
Expired deadlines: 22

Expired records:
                                           Scholarship name Application deadline                                            _source_file
                          Hyundai Women in STEM Scholarship  2026-06-30 00:00:00              Haji-Said_Yahya_National_Scholarships.xlsx
                               GSUSA Gold Award Scholarship  2026-04-14 00:00:00              Haji-Said_Yahya_National_Scholarships.xlsx
             Lindsey Vonn Foundation Enrichment Scholarship  2026-10-31 00:00:00              Haji-Said_Yahya_National_Scholarships.xlsx
                        Women in STEM Scholarship (Learner)  2026-10-19 00:00:00              Haji-Said_Yahya_National_Scholarships.xlsx
                       

In [10]:
# PHASE 2c: INCONSISTENT CATEGORIES

print("=" * 60)
print("INCONSISTENT CATEGORIES")
print("=" * 60)

print("\n - Provider type — RAW values (check capitalization) - ")
print(df['Provider type'].value_counts(dropna=False))

print("\n - Provider type — CLEANED - ")
print(df['provider_type_clean'].value_counts(dropna=False))

print("\n - Renewable — RAW values - ")
print(df['Renewable (Yes/No/Unknown)'].value_counts(dropna=False))

print("\n - Renewable — CLEANED - ")
print(df['Renewable (Yes/No/Unknown)_clean'].value_counts(dropna=False))

print("\n - Essay required — RAW - ")
print(df['Essay required'].value_counts(dropna=False))

name_variants = df['Scholarship name'].str.strip().str.lower().value_counts()
suspicious = name_variants[name_variants > 1]
if len(suspicious):
    print("\n - Names that appear more than once (case-insensitive) - ")
    print(suspicious)
else:
    print("\n✅ No case-insensitive duplicate scholarship names found")

INCONSISTENT CATEGORIES

 - Provider type — RAW values (check capitalization) - 
Provider type
Foundation/nonprofit                46
NaN                                 25
Foundation/Nonprofit                22
Corporation                         17
Private donor / online platform     10
Nonprofit                            8
College/University                   6
Professional association             5
Government                           3
Government agency                    3
Community foundation                 3
Veterans service organization        2
Private donor / scholarship fund     2
Corporation / foundation             1
Corporation / program                1
Public benefit corporation           1
Company                              1
University                           1
Professional association/union       1
Corporate/company foundation         1
Corporate/company program            1
Name: count, dtype: int64

 - Provider type — CLEANED - 
provider_type_clean
Foundatio

In [11]:
# PHASE 2d: INITIAL COMPARISONS

print("=" * 60)
print("COMPARISON 1: SCHOLARSHIPS BY STATE")
print("=" * 60)

state_counts = df['state_clean'].value_counts()
print("\n - Count of scholarships per state - ")
print(state_counts)

print("\n - Award value summary by state - ")
state_stats = df.groupby('state_clean')['award_max_num'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).round(0).sort_values('count', ascending=False)
print(state_stats)

print("\n - State × National vs. state/local crosstab - ")
print(pd.crosstab(df['state_clean'], df['National vs. state/local'], margins=True))

COMPARISON 1: SCHOLARSHIPS BY STATE

 - Count of scholarships per state - 
state_clean
National               48
Indiana                23
Unknown                22
New Mexico             20
California             16
New York               15
Multi-state / Other     5
Puerto Rico             4
New Hampshire           3
Canada                  2
Iowa                    1
Massachusetts           1
Name: count, dtype: int64

 - Award value summary by state - 
                     count     mean   median     min       max
state_clean                                                   
National                35  17644.0   5000.0   500.0  120000.0
California              11  13773.0   5000.0  1000.0   50000.0
New Mexico              11   2464.0   2400.0   250.0    5000.0
Indiana                 10   3360.0   2500.0  1000.0    8000.0
New York                 9  17289.0   5000.0   500.0   80000.0
Multi-state / Other      5  14100.0  10000.0  2000.0   36000.0
Puerto Rico              3  28667.0

In [12]:
# COMPARISON 2: SCHOLARSHIPS BY GRADE LEVEL

print("=" * 60)
print("COMPARISON 2: SCHOLARSHIPS BY GRADE LEVEL")
print("=" * 60)

print("\n - Middle school / high school / both distribution - ")
print(df['Middle school / high school / both'].value_counts(dropna=False))

print("\n - Award value by grade-level category - ")
grade_stats = df.groupby('Middle school / high school / both')['award_max_num'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).round(0)
print(grade_stats)

print("\n - Minimum eligible grade (parsed) - ")
print(df['grade_min'].describe().round(1))

print("\n - Scholarships open to grade 9 or lower - ")
early = df[df['grade_min'] <= 9]
print(f"Count: {len(early)} of {len(df)} ({len(early)/len(df)*100:.1f}%)")

print("\n - Scholarships open to middle school (grade_min <= 8) - ")
middle = df[df['grade_min'] <= 8]
print(f"Count: {len(middle)} of {len(df)} ({len(middle)/len(df)*100:.1f}%)")

print("\n - Grade-level category by source file - ")
print(pd.crosstab(df['_source_file'], df['Middle school / high school / both'], margins=True))

COMPARISON 2: SCHOLARSHIPS BY GRADE LEVEL

 - Middle school / high school / both distribution - 
Middle school / high school / both
High school                                     105
NaN                                              22
Both                                              7
High school (seniors) + college                   7
Middle school                                     5
Both (HS seniors) + college                       4
Both (also elementary)                            3
Both (HS seniors) + college/grad                  2
Both (Middle focus)                               1
Unknown                                           1
Middle school (supports through high school)      1
High school + college/graduate                    1
High school + college (level not specified)       1
Name: count, dtype: int64

 - Award value by grade-level category - 
                                              count     mean   median     min  \
Middle school / high school / both        

In [13]:
# COMPARISON 3: SCHOLARSHIPS BY CATEGORY & FIELD

print("=" * 60)
print("COMPARISON 3: SCHOLARSHIPS BY CATEGORY")
print("=" * 60)

print("\n - Scholarship category distribution - ")
print(df['Scholarship category'].value_counts(dropna=False).head(15))

print("\n - Award value by scholarship category - ")
cat_stats = df.groupby('Scholarship category')['award_max_num'].agg(
    ['count', 'mean', 'median', 'max']
).round(0).sort_values('count', ascending=False).head(10)
print(cat_stats)

print("\n - Field / area of interest — most common - ")
print(df['Field/area of interest'].value_counts(dropna=False).head(15))

COMPARISON 3: SCHOLARSHIPS BY CATEGORY

 - Scholarship category distribution - 
Scholarship category
NaN                                       22
Merit-based                               19
Women-focused / STEM                       7
Need-based                                 6
Career-based                               4
Community service                          3
Skilled trades                             2
General / access                           2
Women-focused / general college access     2
Civic engagement / patriotism              2
Leadership / school access                 2
Leadership / civic engagement              2
Identity + career goal                     2
Career education                           1
Employee scholarship                       1
Name: count, dtype: int64

 - Award value by scholarship category - 
                               count     mean   median       max
Scholarship category                                            
Merit-based              

In [14]:
# COMPARISON 4: AWARD AMOUNTS & DEADLINES

print("=" * 60)
print("COMPARISON 4: AWARD AMOUNT DISTRIBUTION & DEADLINE PATTERNS")
print("=" * 60)

print("\n - Award bucket distribution - ")
print(df['award_bucket'].value_counts())

print("\n - Award bucket by provider type - ")
print(pd.crosstab(df['provider_type_clean'], df['award_bucket'], margins=True))

print("\n - Award bucket by national vs state/local - ")
print(pd.crosstab(df['National vs. state/local'], df['award_bucket'], margins=True))

print("\n - Deadline season distribution - ")
print(df['deadline_season'].value_counts())

print("\n - Deadline month distribution - ")
print(df['deadline_month'].value_counts())

print("\n - Deadline season by grade-level category - ")
print(pd.crosstab(df['Middle school / high school / both'], df['deadline_season'], margins=True))

print("\n - Deadline season by source file - ")
print(pd.crosstab(df['_source_file'], df['deadline_season'], margins=True))

COMPARISON 4: AWARD AMOUNT DISTRIBUTION & DEADLINE PATTERNS

 - Award bucket distribution - 
award_bucket
Unknown            72
$1,000-$4,999      37
$10,000 or more    26
$5,000-$9,999      17
Under $1,000        8
Name: count, dtype: int64

 - Award bucket by provider type - 
award_bucket                      $1,000-$4,999  $10,000 or more  \
provider_type_clean                                                
College/University                            4                0   
Community Foundation                          1                0   
Company                                       0                0   
Corporate/Company Foundation                  0                1   
Corporate/Company Program                     0                1   
Corporation                                   8                8   
Corporation / Foundation                      0                1   
Corporation / Program                         0                0   
Foundation/Nonprofit                     

In [15]:
# OPTIONAL: Save the cleaned dataset for Phase 3

df_out = df.drop(columns=['_name_lower'])
df_out.to_excel('combined_scholarships_cleaned.xlsx', index=False)
print(f"✅ Saved cleaned dataset: combined_scholarships_cleaned.xlsx")
print(f"   Shape: {df_out.shape}")
print(f"   New columns added: award_max_num, award_bucket, grade_min, grade_max,")
print(f"                      deadline_dt, deadline_month, deadline_season,")
print(f"                      *_clean (normalized Yes/No/Unknown),")
print(f"                      provider_type_clean, state_clean, is_expired")

✅ Saved cleaned dataset: combined_scholarships_cleaned.xlsx
   Shape: (160, 47)
   New columns added: award_max_num, award_bucket, grade_min, grade_max,
                      deadline_dt, deadline_month, deadline_season,
                      *_clean (normalized Yes/No/Unknown),
                      provider_type_clean, state_clean, is_expired


# Phase 2 Summary — EDA Findings

## 1. Data Cleaning & Standardization

Seven source files were combined into a single master DataFrame of **160 scholarship records** across the shared 29-column schema (plus a `_source_file` tracking column). The seven files contributed unevenly:

| Source file | Rows | % of cells missing |
|---|---|---|
| Kone_Cheick_National_Scholarships.xlsx | 25 | 14.8% |
| Kone_Cheick_NH_Scholarships.xlsx | 25 | **63.2%** |
| Haji-Said_Yahya_National_Scholarships.xlsx | 18 | 2.0% |
| Haji-Said_Yahya_Indiana_Scholarships.xlsx | 22 | 3.7% |
| Welzant_Kayla_scholarships_NY_leadership_civic_N-Z.xlsx | 15 | 4.1% |
| Lee_Julia_Data.xlsx | 23 | 2.6% |
| Jaibrien_Brooks Scholarship Data.xlsx | 32 | 4.9% |

**Fields standardized:**

| Field | Transformation | Status |
|---|---|---|
| Award amount | Regex-extracted max dollar value → `award_max_num`; bucketed into ranges → `award_bucket` | ✅ Working |
| Grade level | Parsed min/max grade numbers → `grade_min`, `grade_max` | ✅ Working |
| Deadline | Converted to datetime → `deadline_dt`; added `deadline_month`, `deadline_season` | ✅ Working |
| Yes/No/Unknown fields | Regex-prefix normalized → `*_clean` columns | ✅ Working |
| Provider type | Title-cased → `provider_type_clean` | ✅ Working |
| State/region | Extracted primary state → `state_clean` | ✅ Working |

A total of 47 columns are now in the cleaned dataset (29 original + 18 derived).

---

## 2. Summary Statistics

**Dataset size:**
- Total scholarships: **160**
- Source files: **7**
- Unique providers: **140**

**Award amounts (parsed from 88 records — 55% of the dataset had an extractable value):**

| Statistic | Value |
|---|---|
| Mean | **$13,321.89** |
| Median | **$4,000** |
| Std Dev | $22,556.32 |
| Min | $250 (Northern NM College Supercomputing Challenge) |
| Max | **$120,000** (NSA Stokes Educational Scholarship) |
| 25th percentile | $2,000 |
| 75th percentile | $13,000 |

The distribution is **heavily right-skewed** — the mean is more than 3× the median, driven by a handful of very large national awards (NSA Stokes $120,000, Davidson Fellows $100,000, Medical condition scholarship $80,000). The **median ($4,000)** is a much better representation of a typical scholarship than the mean.

**Award amount by state:**

| State | Count | Mean | Median |
|---|---|---|---|
| National | 35 | $17,644 | $5,000 |
| California | 11 | $13,773 | $5,000 |
| New Mexico | 11 | $2,464 | $2,400 |
| Indiana | 10 | $3,360 | $2,500 |
| New York | 9 | $17,289 | $5,000 |
| Multi-state / Other | 5 | $14,100 | $10,000 |
| Puerto Rico | 3 | $28,667 | $25,000 |
| Canada | 2 | $13,750 | $13,750 |
| Iowa | 1 | $1,000 | $1,000 |
| Massachusetts | 1 | $2,000 | $2,000 |

**Award amount by source file (median):**

| File | Count | Median | Max |
|---|---|---|---|
| Haji-Said_Indiana | 9 | $2,500 | $8,000 |
| Haji-Said_National | 16 | $5,000 | $25,000 |
| Jaibrien_Brooks | 17 | $5,000 | **$100,000** |
| Kone_National | 23 | $5,000 | $80,000 |
| Lee_Julia (NM) | 14 | $2,450 | **$120,000** |
| Welzant_NY | 9 | $5,000 | $40,000 |
| Kone_NH | 0 | — | — (no parseable award amounts) |

---

## 3. Frequency Distributions

**Provider type:** Dominated by **Foundation/Nonprofit (68 records, 42.5%)**, followed by Corporation (17, 10.6%), Private Donor / Online Platform (10, 6.2%), Nonprofit (8, 5.0%), and College/University (6, 3.8%). **25 records have no provider type at all** — mostly the New Hampshire file.

**National vs. state/local:**
- State/local: **83 records (51.9%)**
- National: **64 records (40.0%)**
- Local: 9 (5.6%)
- A few specialized tags: "National (state-level winners)" (2), "State" (1), "National program, state-selected" (1)

**Grade level:**
- High school only: **105 records (65.6%)**
- Both: 7 (4.4%)
- Middle school only: **5 records (3.1%)**
- 22 records have no grade-level classification

**Award bucket:**

| Bucket | Count | % |
|---|---|---|
| Unknown | 72 | 45.0% |
| $1,000–$4,999 | 37 | 23.1% |
| $10,000 or more | 26 | 16.2% |
| $5,000–$9,999 | 17 | 10.6% |
| Under $1,000 | 8 | 5.0% |

**Deadline season (only 58 records had a parseable deadline):**
- Unknown / not posted: **102 (63.7%)**
- Winter (Dec–Feb): **28 (17.5%)**
- Spring (Mar–May): **22 (13.8%)**
- Fall (Sep–Nov): 5 (3.1%)
- Summer (Jun–Aug): 3 (1.9%)

**Deadline months** show a clear pattern: **January (14) and March (14) are the busiest deadline months**, followed by December (7), February (7), and April (6). This confirms the typical late-winter/early-spring application cycle.

**Yes/No fields (after normalization):**

| Field | Yes | No | Unknown |
|---|---|---|---|
| Essay required | 55 | 8 | 97 |
| Recommendation required | 45 | 8 | 107 |
| Financial need required | 33 | 29 | 98 |
| Renewable | 24 | 46 | 90 |

Essay is required by ~34% of scholarships, recommendations by ~28%, and financial need matters for ~21%. Renewal is explicitly denied in 46 records (28.7%) and confirmed in 24 (15%), but the majority (56%) remain unclear.

---

## 4. Data Quality Findings

**Missing values:** About **14.7% of all cells are missing** (1,056 missing cells out of 7,200). **13 of 45 columns are 100% complete.** The most-missing fields are:

| Column | Missing | % Missing |
|---|---|---|
| deadline_dt | 102 | 63.7% |
| deadline_month | 102 | 63.7% |
| award_max_num | 72 | 45.0% |
| grade_min / grade_max | 58 | 36.2% |
| Application open date (if known) | 45 | 28.1% |
| Minimum GPA (if any) | 42 | 26.2% |
| Application deadline | 41 | 25.6% |
| Number of awards (if known) | 33 | 20.6% |
| Maximum potential value | 27 | 16.9% |

**Missingness by source file:** The **New Hampshire file is by far the sparsest (63.2%)** — 24 of 25 rows are research leads with only name, URL, and state filled in, and **none produced a parseable award amount**. The other six files are much cleaner, clustered at 2–15% missing. This is the main data-collection inconsistency in the combined set.

**Duplicates:** ✅ Excellent news — **no exact duplicate rows, no duplicate scholarship names (case-insensitive), and no scholarships appearing in multiple source files.** The seven files are genuinely non-overlapping.

**Expired listings:** **22 of 58 dated records have already expired** (deadline is before today, 2026-11-08). This is the single most important Phase 3 concern — nearly 40% of the dated deadlines are already in the past. Expired examples span all files:

- Hyundai Women in STEM (2026-06-30)
- GSUSA Gold Award (2026-04-14)
- Amy Hasbrook Leadership Award (2026-02-13)
- Priscilla Dillow Seize the Day (2026-02-13)
- Davis-Kozoll Scholarship (2026-04-02)
- VFW Patriot's Pen / Voice of Democracy (2026-10-31)
- National Space Club Keynote Scholar (2026-10-30)
- ... and 15 others

**36 of 58 dated deadlines are still upcoming** — these are the actionable ones for students this cycle.

**Inconsistent categories:**
- **Provider type** had heavy capitalization and format variance: "Foundation/nonprofit" (46) vs "Foundation/Nonprofit" (22), "Private donor / online platform" vs "Private Donor / Online Platform", "Corporation / foundation" vs "Corporate/company foundation". The `.title()` cleanup collapsed these 19 raw labels into 19 cleaned labels — several are still conceptually overlapping ("Company", "Corporation", "Corporate/Company Foundation") and should be merged in Phase 3.
- **Renewable** was the worst offender — 22 distinct raw formats (`"Yes"`, `"Yes (up to 4 years)"`, `"Yes (service commitment)"`, `"No (paid over four years)"`, `"No (the $1,000 is invested until high school graduation)"`, etc.). These are now correctly normalized into `Yes` / `No` / `Unknown` using regex prefix matching.
- **Essay required** had 30+ variants like `"Yes (≤500 words)"`, `"Yes (essay and statement, each ≤500 words)"`, `"No (video presentation instead)"` — now normalized into `Yes` / `No` / `Unknown`.
- **Grade level** in the raw `Middle school / high school / both` column has 13 distinct values (`"High school"`, `"Both"`, `"Both (HS seniors) + college"`, `"Both (also elementary)"`, `"Unknown"`, etc.). The **derived** `grade_min` / `grade_max` columns now correctly parse ranges like `"11–12"` → `(11, 12)`.
- **State extraction** now works correctly after fixing the missing commas.

---

## 5. Initial Comparisons

**By state:**
- National: 48
- Indiana: 23
- Unknown: 22 (NH research leads with no state field)
- New Mexico: 20
- California: 16
- New York: 15
- Multi-state / Other: 5
- Puerto Rico: 4
- New Hampshire: 3
- Canada: 2
- Iowa: 1
- Massachusetts: 1

This shows the true geographic distribution across the seven files.

**By grade level:** **105 of 160 (65.6%) are high-school-only**, with middle-school-specific opportunities being rare (only 5 records). The `grade_min` distribution is now realistic — median 12.0 (senior year), mean 10.5, with 22 records open to grade 9 or lower (13.8%) and 17 records open to middle school (10.6%). This is a known reality of the scholarship landscape: **most scholarships target graduating high-school seniors**.

**By category:** The most common category is **Merit-based (19 records)**, followed by Women-focused / STEM (7), Need-based (6), Career-based (4), and Community service (3). 22 records have no category at all. Award value by category:
- Merit-based: median $2,500, max $120,000
- Women-focused / STEM: median $3,000, max $20,000
- Civic engagement / patriotism: median $20,000 (small sample)

**By award amount:** The distribution is **extremely right-skewed** — median $4,000, mean $13,322, standard deviation $22,556 (nearly double the mean). The award bucket breakdown shows that of the 88 records with a parseable value:
- Under $1,000: 8 (9%)
- $1,000–$4,999: 37 (42%)
- $5,000–$9,999: 17 (19%)
- $10,000 or more: 26 (30%)

So **most scholarships (51%) fall under $5,000**, but a small set of high-value national awards pulls the mean far above the median. The California file's many $1,000–$5,000 awards balance the huge outliers from the NM file, making the **median a far better representation of a typical scholarship than the mean**.

**By deadline:** With 58 records having parseable deadlines, **Winter (28) and Spring (22)** dominate. **January (14) and March (14) are the single busiest months**. This confirms that the **first quarter of the calendar year** is when most scholarship applications are due — actionable guidance for students and counselors planning the search.

---

## 6. Recommendations for Phase 3

1. **Re-verify expired deadlines** — 22 of 58 dated records are already past. Phase 3 visualizations and recommendations should either filter to upcoming only or clearly label stale deadlines as "prior cycle — verify next year."

2. **Merge overlapping provider categories** — "Company", "Corporation", "Corporate/Company Foundation", and "Corporate/Company Program" should collapse into a single "Corporation" bucket. Similarly, "Nonprofit" and "Foundation/Nonprofit" should merge into one.

3. **De-emphasize the New Hampshire file** in any quantitative conclusions — 63.2% missing, no parseable award amounts. It's a research-lead file, not a verified dataset.

4. **Flag LOW CONFIDENCE rows** — the Collie Future Educators and Gussie Lynn scholarships (from the Jaibrien_Brooks file) were already marked as low-confidence in the original research notes and should be excluded from any dollar-value claims.

5. **Investigate outlier awards as a sub-group** — the NSA Stokes ($120k), Davidson Fellows ($100k), and similar high-value awards are qualitatively different from typical $2k–$5k scholarships. A separate "high-value national awards" group would make comparisons fairer.

6. **Analyze the "Future Teacher" cluster separately** — the California file adds 15+ teacher-education-specific awards, which is a rich sub-category worth its own Phase 3 analysis.

7. **Add visualizations** for Phase 3:
   - Bar chart: scholarship count per state
   - Histogram: award amount distribution (consider log-scale or truncation at $50k to show shape without outlier dominance)
   - Box plot: award amount by provider type
   - Bar chart: deadline month distribution (showing the Jan/Mar peaks)
   - Pie or stacked bar: Middle school vs High school vs Both